In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("set3").getOrCreate()



# Scenario 1: Inventory Alerting System

In [0]:
# Q1. Load the data using PySpark
df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Shared/inventory_supply.csv")
df.show()


+------+------------+-----------+----------+--------+------------+-------------+---------+---------+
|ItemID|    ItemName|   Category| Warehouse|StockQty|ReorderLevel|LastRestocked|UnitPrice| Supplier|
+------+------------+-----------+----------+--------+------------+-------------+---------+---------+
|  I001|      LED TV|Electronics|WarehouseA|      50|          20|   2024-03-15|    30000|   AVTech|
|  I002|      Laptop|Electronics|WarehouseB|      10|          15|   2024-04-01|    70000|TechWorld|
|  I003|Office Chair|  Furniture|WarehouseA|      40|          10|   2024-03-25|     6000|  ChairCo|
|  I004|Refrigerator| Appliances|WarehouseC|       5|          10|   2024-02-20|    25000| FreezeIt|
|  I005|     Printer|Electronics|WarehouseB|       3|           5|   2024-03-30|     8000|PrintFast|
+------+------------+-----------+----------+--------+------------+-------------+---------+---------+



In [0]:
# Q2. Add column 'NeedsReorder' to indicate whether an item needs restocking
from pyspark.sql.functions import col

df = df.withColumn("NeedsReorder", col("StockQty") < col("ReorderLevel"))
df.show()


+------+------------+-----------+----------+--------+------------+-------------+---------+---------+------------+
|ItemID|    ItemName|   Category| Warehouse|StockQty|ReorderLevel|LastRestocked|UnitPrice| Supplier|NeedsReorder|
+------+------------+-----------+----------+--------+------------+-------------+---------+---------+------------+
|  I001|      LED TV|Electronics|WarehouseA|      50|          20|   2024-03-15|    30000|   AVTech|       false|
|  I002|      Laptop|Electronics|WarehouseB|      10|          15|   2024-04-01|    70000|TechWorld|        true|
|  I003|Office Chair|  Furniture|WarehouseA|      40|          10|   2024-03-25|     6000|  ChairCo|       false|
|  I004|Refrigerator| Appliances|WarehouseC|       5|          10|   2024-02-20|    25000| FreezeIt|        true|
|  I005|     Printer|Electronics|WarehouseB|       3|           5|   2024-03-30|     8000|PrintFast|        true|
+------+------------+-----------+----------+--------+------------+-------------+--------

In [0]:
# Q3. Create view of items needing restock
df.filter("NeedsReorder = true").createOrReplaceTempView("items_to_restock")

# Display the view
spark.sql("SELECT * FROM items_to_restock").show()


+------+------------+-----------+----------+--------+------------+-------------+---------+---------+------------+
|ItemID|    ItemName|   Category| Warehouse|StockQty|ReorderLevel|LastRestocked|UnitPrice| Supplier|NeedsReorder|
+------+------------+-----------+----------+--------+------------+-------------+---------+---------+------------+
|  I002|      Laptop|Electronics|WarehouseB|      10|          15|   2024-04-01|    70000|TechWorld|        true|
|  I004|Refrigerator| Appliances|WarehouseC|       5|          10|   2024-02-20|    25000| FreezeIt|        true|
|  I005|     Printer|Electronics|WarehouseB|       3|           5|   2024-03-30|     8000|PrintFast|        true|
+------+------------+-----------+----------+--------+------------+-------------+---------+---------+------------+



In [0]:
# Q4. Identify warehouses with more than 2 restocking items
spark.sql("""
    SELECT Warehouse, COUNT(*) as RestockItemCount
    FROM items_to_restock
    GROUP BY Warehouse
    HAVING COUNT(*) > 2
""").show()


+---------+----------------+
|Warehouse|RestockItemCount|
+---------+----------------+
+---------+----------------+



# Scenario 2: Supplier Price Optimization

In [0]:
# Scenario 2 - Q1: Average UnitPrice per Supplier
df.groupBy("Supplier").avg("UnitPrice").withColumnRenamed("avg(UnitPrice)", "AvgPrice").show()


+---------+--------+
| Supplier|AvgPrice|
+---------+--------+
|   AVTech| 30000.0|
|TechWorld| 70000.0|
|PrintFast|  8000.0|
| FreezeIt| 25000.0|
|  ChairCo|  6000.0|
+---------+--------+



In [0]:
# Scenario 2 - Q2: Items priced below category average
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

cat_avg = df.withColumn("CategoryAvg", avg("UnitPrice").over(Window.partitionBy("Category")))
df_below_avg = cat_avg.filter(col("UnitPrice") < col("CategoryAvg"))
df_below_avg.select("ItemName", "Category", "Supplier", "UnitPrice", "CategoryAvg").show()


+--------+-----------+---------+---------+-----------+
|ItemName|   Category| Supplier|UnitPrice|CategoryAvg|
+--------+-----------+---------+---------+-----------+
|  LED TV|Electronics|   AVTech|    30000|    36000.0|
| Printer|Electronics|PrintFast|     8000|    36000.0|
+--------+-----------+---------+---------+-----------+



In [0]:
# Scenario 2 - Q3: Tag Good Deal suppliers
below_df = df_below_avg.groupBy("Supplier").count().withColumnRenamed("count", "BelowAvgCount")
total_df = df.groupBy("Supplier").count().withColumnRenamed("count", "TotalCount")

good_deal_df = below_df.join(total_df, "Supplier")
good_deal_df = good_deal_df.withColumn("GoodDeal", (col("BelowAvgCount") / col("TotalCount")) > 0.5)
good_deal_df.select("Supplier", "GoodDeal").show()


+---------+--------+
| Supplier|GoodDeal|
+---------+--------+
|   AVTech|    true|
|PrintFast|    true|
+---------+--------+



#  Scenario 3: Cost Forecasting

In [0]:
# Scenario 3 - Q1: Total stock value
df = df.withColumn("TotalStockValue", col("StockQty") * col("UnitPrice"))
df.select("ItemID", "ItemName", "StockQty", "UnitPrice", "TotalStockValue").show()


+------+------------+--------+---------+---------------+
|ItemID|    ItemName|StockQty|UnitPrice|TotalStockValue|
+------+------------+--------+---------+---------------+
|  I001|      LED TV|      50|    30000|        1500000|
|  I002|      Laptop|      10|    70000|         700000|
|  I003|Office Chair|      40|     6000|         240000|
|  I004|Refrigerator|       5|    25000|         125000|
|  I005|     Printer|       3|     8000|          24000|
+------+------------+--------+---------+---------------+



In [0]:
# Scenario 3 - Q2: Top 3 highest-value items
df.orderBy(col("TotalStockValue").desc()).select("ItemID", "ItemName", "TotalStockValue").show(3)


+------+------------+---------------+
|ItemID|    ItemName|TotalStockValue|
+------+------------+---------------+
|  I001|      LED TV|        1500000|
|  I002|      Laptop|         700000|
|  I003|Office Chair|         240000|
+------+------------+---------------+
only showing top 3 rows



In [0]:
# Scenario 3 - Q3: Write Parquet partitioned by Warehouse
df.write.mode("overwrite").partitionBy("Warehouse").parquet("/mnt/data/output/inventory_by_warehouse")


# Scenario 4: Warehouse Utilization

In [0]:
# Scenario 4 - Q1: Items per warehouse
df.groupBy("Warehouse").count().withColumnRenamed("count", "ItemCount").show()


+----------+---------+
| Warehouse|ItemCount|
+----------+---------+
|WarehouseA|        2|
|WarehouseC|        1|
|WarehouseB|        2|
+----------+---------+



In [0]:
# Scenario 4 - Q2: Average stock by Category and Warehouse
df.groupBy("Warehouse", "Category").avg("StockQty").withColumnRenamed("avg(StockQty)", "AvgStock").show()


+----------+-----------+--------+
| Warehouse|   Category|AvgStock|
+----------+-----------+--------+
|WarehouseB|Electronics|     6.5|
|WarehouseA|  Furniture|    40.0|
|WarehouseC| Appliances|     5.0|
|WarehouseA|Electronics|    50.0|
+----------+-----------+--------+



In [0]:
# Scenario 4 - Q3: Underutilized warehouses
df.groupBy("Warehouse").sum("StockQty") \
  .withColumnRenamed("sum(StockQty)", "TotalStock") \
  .filter("TotalStock < 100").show()


+----------+----------+
| Warehouse|TotalStock|
+----------+----------+
|WarehouseA|        90|
|WarehouseC|         5|
|WarehouseB|        13|
+----------+----------+



# Scenario 5: Delta Audit Trail

In [0]:
# Scenario 5 - Q1: Save as Delta table
df.write.format("delta").mode("overwrite").saveAsTable("retail_inventory")


In [0]:
# Scenario 5 - Q2: Update stock using SQL
spark.sql("""
  UPDATE retail_inventory
  SET StockQty = 20
  WHERE ItemName = 'Laptop'
""")


DataFrame[num_affected_rows: bigint]

In [0]:
# Scenario 5 - Q3: Delete zero stock items
spark.sql("""
  DELETE FROM retail_inventory
  WHERE StockQty = 0
""")


DataFrame[num_affected_rows: bigint]

In [0]:
# Scenario 5 - Q4: Audit delta history
spark.sql("DESCRIBE HISTORY retail_inventory").show()


+-------+-------------------+----------------+--------------------+--------------------+--------------------+----+------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|          userId|            userName|           operation| operationParameters| job|          notebook|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+----------------+--------------------+--------------------+--------------------+----+------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      3|2025-06-19 05:05:53|7928277367239535|azuser3564_mml.lo...|              DELETE|{predicate -> ["(...|NULL|{2805786059712361}|0611-043414-4p180ssa|          2|WriteSerializable|        false|{numRemovedFiles ...|        NULL|Databr

# Scenario 6: Alerts from Restock Logs

In [0]:
# Scenario 6 - Q1: Load restock_logs and join
logs_df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Shared/restock_logs.csv")
joined_df = df.join(logs_df, "ItemID", "left") \
              .withColumn("NewStockQty", col("StockQty") + col("QuantityAdded"))


In [0]:
# Scenario 6 - Q2: Add flag for restocked items
from pyspark.sql.functions import when
joined_df = joined_df.withColumn("RestockedRecently", when(col("QuantityAdded").isNotNull(), True).otherwise(False))
joined_df.select("ItemID", "ItemName", "StockQty", "QuantityAdded", "NewStockQty", "RestockedRecently").show()


+------+------------+--------+-------------+-----------+-----------------+
|ItemID|    ItemName|StockQty|QuantityAdded|NewStockQty|RestockedRecently|
+------+------------+--------+-------------+-----------+-----------------+
|  I001|      LED TV|      50|           20|         70|             true|
|  I002|      Laptop|      10|           10|         20|             true|
|  I005|     Printer|       3|            5|          8|             true|
|  I003|Office Chair|      40|         NULL|       NULL|            false|
|  I004|Refrigerator|       5|         NULL|       NULL|            false|
+------+------------+--------+-------------+-----------+-----------------+



In [0]:
# Scenario 6 - Q3: MERGE into Delta table
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "retail_inventory")

target.alias("t").merge(
    joined_df.alias("s"),
    "t.ItemID = s.ItemID AND s.QuantityAdded IS NOT NULL"
).whenMatchedUpdate(set={"StockQty": "s.NewStockQty"}).execute()


# Scenario 7: Report Generation with SQL Views

In [0]:
# Scenario 7 - Q1: Create inventory summary view
df.createOrReplaceTempView("inventory_summary")
spark.sql("""
  SELECT ItemName, Category, StockQty, NeedsReorder, TotalStockValue
  FROM inventory_summary
""").show()


+------------+-----------+--------+------------+---------------+
|    ItemName|   Category|StockQty|NeedsReorder|TotalStockValue|
+------------+-----------+--------+------------+---------------+
|      LED TV|Electronics|      50|       false|        1500000|
|      Laptop|Electronics|      10|        true|         700000|
|Office Chair|  Furniture|      40|       false|         240000|
|Refrigerator| Appliances|       5|        true|         125000|
|     Printer|Electronics|       3|        true|          24000|
+------------+-----------+--------+------------+---------------+



In [0]:
# Scenario 7 - Q2: Supplier leaderboard by avg price
df.groupBy("Supplier").avg("UnitPrice").withColumnRenamed("avg(UnitPrice)", "AvgPrice") \
  .createOrReplaceTempView("supplier_leaderboard")

spark.sql("SELECT * FROM supplier_leaderboard ORDER BY AvgPrice").show()


+---------+--------+
| Supplier|AvgPrice|
+---------+--------+
|  ChairCo|  6000.0|
|PrintFast|  8000.0|
| FreezeIt| 25000.0|
|   AVTech| 30000.0|
|TechWorld| 70000.0|
+---------+--------+



# Scenario 8: Advanced Filtering

In [0]:
# Scenario 8 - Q1: Categorize stock level
df = df.withColumn("StockStatus", when(col("StockQty") > 2 * col("ReorderLevel"), "Overstocked")
                   .when(col("StockQty") < col("ReorderLevel"), "LowStock")
                   .otherwise("Normal"))
df.select("ItemName", "StockQty", "ReorderLevel", "StockStatus").show()


+------------+--------+------------+-----------+
|    ItemName|StockQty|ReorderLevel|StockStatus|
+------------+--------+------------+-----------+
|      LED TV|      50|          20|Overstocked|
|      Laptop|      10|          15|   LowStock|
|Office Chair|      40|          10|Overstocked|
|Refrigerator|       5|          10|   LowStock|
|     Printer|       3|           5|   LowStock|
+------------+--------+------------+-----------+



In [0]:
# Scenario 8 - Q2: Use filter()
df.filter(col("StockStatus") == "LowStock").show()

# Use where()
df.where(col("StockStatus") == "LowStock").show()


+------+------------+-----------+----------+--------+------------+-------------+---------+---------+------------+---------------+-----------+
|ItemID|    ItemName|   Category| Warehouse|StockQty|ReorderLevel|LastRestocked|UnitPrice| Supplier|NeedsReorder|TotalStockValue|StockStatus|
+------+------------+-----------+----------+--------+------------+-------------+---------+---------+------------+---------------+-----------+
|  I002|      Laptop|Electronics|WarehouseB|      10|          15|   2024-04-01|    70000|TechWorld|        true|         700000|   LowStock|
|  I004|Refrigerator| Appliances|WarehouseC|       5|          10|   2024-02-20|    25000| FreezeIt|        true|         125000|   LowStock|
|  I005|     Printer|Electronics|WarehouseB|       3|           5|   2024-03-30|     8000|PrintFast|        true|          24000|   LowStock|
+------+------------+-----------+----------+--------+------------+-------------+---------+---------+------------+---------------+-----------+

+----

# Scenario 9: Feature Engineering

In [0]:
# Scenario 9 - Q1: Extract RestockMonth
from pyspark.sql.functions import month

df = df.withColumn("RestockMonth", month("LastRestocked"))
df.select("ItemName", "LastRestocked", "RestockMonth").show()


+------------+-------------+------------+
|    ItemName|LastRestocked|RestockMonth|
+------------+-------------+------------+
|      LED TV|   2024-03-15|           3|
|      Laptop|   2024-04-01|           4|
|Office Chair|   2024-03-25|           3|
|Refrigerator|   2024-02-20|           2|
|     Printer|   2024-03-30|           3|
+------------+-------------+------------+



In [0]:
# Scenario 9 - Q2: Calculate stock age
from pyspark.sql.functions import current_date, datediff

df = df.withColumn("StockAge", datediff(current_date(), col("LastRestocked")))
df.select("ItemName", "LastRestocked", "StockAge").show()


+------------+-------------+--------+
|    ItemName|LastRestocked|StockAge|
+------------+-------------+--------+
|      LED TV|   2024-03-15|     461|
|      Laptop|   2024-04-01|     444|
|Office Chair|   2024-03-25|     451|
|Refrigerator|   2024-02-20|     485|
|     Printer|   2024-03-30|     446|
+------------+-------------+--------+



In [0]:
# Scenario 9 - Q3: Bucket stock age
df = df.withColumn("StockAgeBucket", when(col("StockAge") < 30, "New")
                   .when((col("StockAge") >= 30) & (col("StockAge") <= 90), "Moderate")
                   .otherwise("Stale"))
df.select("ItemName", "StockAge", "StockAgeBucket").show()


+------------+--------+--------------+
|    ItemName|StockAge|StockAgeBucket|
+------------+--------+--------------+
|      LED TV|     461|         Stale|
|      Laptop|     444|         Stale|
|Office Chair|     451|         Stale|
|Refrigerator|     485|         Stale|
|     Printer|     446|         Stale|
+------------+--------+--------------+



# Scenario 10: Export Options

In [0]:
# Scenario 10 - Q1: Write to CSV
df.write.mode("overwrite").csv("/export/inventory/full_csv", header=True)

# Write to JSON
df.write.mode("overwrite").json("/export/inventory/full_json")

# Write to Delta
df.write.format("delta").mode("overwrite").save("/export/inventory/full_delta")


In [0]:
# Scenario 10 - Q2: Save stale items separately
df.filter(col("StockAgeBucket") == "Stale") \
  .write.mode("overwrite").partitionBy("Warehouse") \
  .parquet("/export/inventory/stale_items")
